# H3 Hexagon Grid Generation: Berlin

This notebook generates an H3 hexagon grid covering Berlin used as the spatial unit for the rest of this thesis's geospatial machine learning pipeline.
Berlin's boundary is loaded from a pre-saved CSV (WKT format) rather than fetched live from OpenStreetMap, for reproducibility.

**Output:** a GeoJSON file of resolution-8 H3 hexagons covering Berlin saved to `data/external/`

In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon
from shapely import wkt
import h3

In [2]:
# Load berlin's boundary
boundary_df = pd.read_csv("../data/external/berlin_boundary.csv")

print(boundary_df.shape)
print(boundary_df.columns.tolist())
boundary_df.head()

(1, 5)
['local_id', 'region', 'city', 'district', 'location']


,local_id,region,city,district,location
0,0,berlin,berlin,berlin,MULTIPOLYGON (((13.500218389999695 52.61379249...


In [3]:
boundary_wkt_text = boundary_df["location"].iloc[0]

# Convert it from text into usable geometric shape
berlin_boundary = wkt.loads(boundary_wkt_text)


print(berlin_boundary.geom_type)
print(len(berlin_boundary.geoms))

MultiPolygon
1


In [4]:

if berlin_boundary.geom_type == 'MultiPolygon':
    polygons = list(berlin_boundary.geoms)
else:
    polygons = [berlin_boundary]

print(f"Number of polygon pieces to process: {len(polygons)}")

Number of polygon pieces to process: 1


## Generate H3 hexagons (resolution 8)

Resolution 8 hexagons are generated as the primary spatial unit. H3 expects coordinates as (latitude, longitude), while Shapely stores them as (longitude, latitude) - coordinates are flipped before being passed to H3.

Only each polygon's exterior ring is used. Berlin's boundary as loaded is a simple outline with no such holes, so this doesn't affect the result, but it's a limitation.

In [5]:
resolution = 8
hex_cells = set()

for poly in polygons:
    coords = [(lat, lon) for lon, lat in poly.exterior.coords]
    h3_poly = h3.LatLngPoly(coords)
    cells = h3.polygon_to_cells(h3_poly, resolution)
    hex_cells.update(cells)

print(f"Number of resolution-{resolution} hexagons generated: {len(hex_cells)}")

Number of resolution-8 hexagons generated: 1353


In [6]:
hex_ids = sorted(hex_cells) 
hex_geometries = []

for cell in hex_ids:
    boundary_coords = h3.cell_to_boundary(cell)
    lon_lat_coords = [(lon, lat) for lat, lon in boundary_coords]
    hex_geometries.append(Polygon(lon_lat_coords))

print(f"Number of shapes built: {len(hex_geometries)}")

Number of shapes built: 1353


In [7]:
hex_gdf = gpd.GeoDataFrame(
    {"h3_index": hex_ids, "resolution": resolution},
    geometry=hex_geometries,
    crs="EPSG:4326"
)

hex_gdf.to_file("../data/external/berlin_h3_res8.geojson", driver="GeoJSON")

print(f"Saved {len(hex_gdf)} hexagons to berlin_h3_res8.geojson")
hex_gdf.head()

Saved 1353 hexagons to berlin_h3_res8.geojson


,h3_index,resolution,geometry
0,881f188401fffff,8,"POLYGON ((13.16441 52.41029, 13.16222 52.40604..."
1,881f188409fffff,8,"POLYGON ((13.16131 52.4175, 13.15911 52.41325,..."
2,881f18840bfffff,8,"POLYGON ((13.17411 52.41584, 13.17191 52.41159..."
3,881f18840dfffff,8,"POLYGON ((13.15162 52.41195, 13.14942 52.4077,..."
4,881f188419fffff,8,"POLYGON ((13.1966 52.41972, 13.19441 52.41547,..."


## Result

Generated 1,353 resolution-8 hexagons covering Berlin, saved to
`berlin_h3_res8.geojson`.